### Google authentication and Imports

In [1]:
# --- STEP 1: install packages, then restart the runtime ---
# Run this notebook on Google Colab (Python 3.12) so precompiled wheels are used.
# After this cell finishes, the runtime restarts automatically; then run the
# next cell (Step 2) to do the patch, imports, and BigQuery auth.
%pip install "spacy>=3.8,<4.0"
%pip install "scispacy==0.6.2"
%pip install https://s3-us-west-2.amazonaws.com/ai2-s2-scispacy/releases/v0.5.4/en_core_sci_sm-0.5.4.tar.gz
%pip install medspacy
%pip install gensim

# Confirm the model actually installed before restarting.
import importlib.util
assert importlib.util.find_spec("en_core_sci_sm") is not None, \
    "en_core_sci_sm did not install correctly -- check the pip output above for errors."
print("en_core_sci_sm is installed. Restarting runtime...")

import os
os.kill(os.getpid(), 9)

  Using cached spacy-3.8.14-cp312-cp312-manylinux2014_x86_64.manylinux_2_17_x86_64.whl.metadata (28 kB)
  Using cached thinc-8.3.13-cp312-cp312-manylinux2014_x86_64.manylinux_2_17_x86_64.whl.metadata (14 kB)
  Using cached weasel-1.0.0-py3-none-any.whl.metadata (4.6 kB)
  Using cached confection-1.3.3-py3-none-any.whl.metadata (19 kB)
  Using cached blis-1.3.3-cp312-cp312-manylinux2014_x86_64.manylinux_2_17_x86_64.whl.metadata (7.5 kB)
Using cached spacy-3.8.14-cp312-cp312-manylinux2014_x86_64.manylinux_2_17_x86_64.whl (33.2 MB)
Using cached confection-1.3.3-py3-none-any.whl (35 kB)
Using cached thinc-8.3.13-cp312-cp312-manylinux2014_x86_64.manylinux_2_17_x86_64.whl (3.9 MB)
Using cached weasel-1.0.0-py3-none-any.whl (50 kB)
Using cached blis-1.3.3-cp312-cp312-manylinux2014_x86_64.manylinux_2_17_x86_64.whl (11.4 MB)
  Attempting uninstall: confection
    Found existing installation: confection 0.1.5
    Uninstalling confection-0.1.5:
      Successfully uninstalled confection-0.1.5
  At

: 

: 

: 

In [ ]:
# --- STEP 2: run AFTER the runtime restart from Step 1 ---
# Patch the en_core_sci_sm 0.5.4 config so spaCy 3.8 accepts it, then import
# everything and authenticate to BigQuery.
import os, re, en_core_sci_sm

model_dir = os.path.dirname(en_core_sci_sm.__file__)
pattern = re.compile(r'include_static_vectors\s*=\s*"?(False|True)"?', re.IGNORECASE)
patched = []
for root, _, files in os.walk(model_dir):
    for fname in files:
        if fname == "config.cfg":
            p = os.path.join(root, fname)
            with open(p, "r", encoding="utf-8") as fh:
                text = fh.read()
            new_text, n = pattern.subn(
                lambda m: f"include_static_vectors = {m.group(1).lower()}", text
            )
            if n and new_text != text:
                with open(p, "w", encoding="utf-8") as fh:
                    fh.write(new_text)
                patched.append(p)
print("Patched config files:", patched)

from google.colab import auth
from google.cloud import bigquery
import scispacy
import medspacy
import spacy
import gensim
import gensim.downloader as api
from sklearn.manifold import TSNE
import matplotlib.pyplot as plt
from gensim.models import Word2Vec

auth.authenticate_user()
project_id = 'ai-on-healthcare'
client = bigquery.Client(project=project_id)

print("spacy:", spacy.__version__, "| scispacy:", scispacy.__version__, "| medspacy:", medspacy.__version__)

### Diagnoses Query

In [ ]:
ICD_FILTER = ['430']

diagnose_query = """
    SELECT *
    FROM `physionet-data.mimiciii_clinical.diagnoses_icd`
    LIMIT 100000
"""

# Run the query and convert the results to a Pandas DataFrame
df_diagnoses = client.query(diagnose_query).to_dataframe()
df_diagnoses = df_diagnoses[df_diagnoses['ICD9_CODE'].isin(ICD_FILTER)]
subj_filtered = df_diagnoses['SUBJECT_ID'].unique()
print(subj_filtered)

### Get notes data

In [ ]:
notes_query = f"""
    SELECT SUBJECT_ID, HADM_ID, CATEGORY, TEXT
    FROM `physionet-data.mimiciii_notes.noteevents`
    WHERE SUBJECT_ID IN UNNEST(@subject_ids)
"""

job_config = bigquery.QueryJobConfig(
    query_parameters=[
        bigquery.ArrayQueryParameter(
            "subject_ids", "INT64", [int(s) for s in subj_filtered]
        )
    ]
)

df_notes = client.query(notes_query, job_config=job_config).to_dataframe()
df_notes_filtered = df_notes.dropna(subset=["TEXT"]).reset_index(drop=True)
print(f"Notes retrieved: {len(df_notes_filtered)} for {df_notes_filtered['SUBJECT_ID'].nunique()} subjects")
df_notes_filtered.head()

### Get pretrained model

In [ ]:
info = api.info()  # show info about available models/datasets
pretrained_model= api.load("glove-wiki-gigaword-50")  # download the model and return as object ready for use

### Function to graph T-SNE

In [ ]:
import numpy as np

def tsne_plot(model,words, preTrained=False):
    "Creates and TSNE model and plots it"
    labels = []
    tokens = []

    for word in words:
      if preTrained:
          tokens.append(model[word])
      else:
          tokens.append(model.wv[word])
      labels.append(word)

    tokens = np.array(tokens)
    tsne_model = TSNE(perplexity=30, early_exaggeration=12, n_components=2, init='pca', n_iter=1000, random_state=23)
    new_values = tsne_model.fit_transform(tokens)

    x = []
    y = []
    for value in new_values:
        x.append(value[0])
        y.append(value[1])

    plt.figure(figsize=(16, 16))
    for i in range(len(x)):
        plt.scatter(x[i],y[i])
        plt.annotate(labels[i],
                     xy=(x[i], y[i]),
                     xytext=(5, 2),
                     textcoords='offset points',
                     ha='right',
                     va='bottom')
    plt.show()

### Extract Entities with Spacy

In [ ]:
nlp = spacy.load("en_core_web_sm")
corpus = []
for row in range(0, len(df_notes_filtered)):
    ents = nlp(df_notes_filtered.iloc[row]['TEXT']).ents
    str_tokens = [t.text for t in ents]
    if str_tokens:  # skip notes with no entities
        corpus.append(str_tokens)

print(f"Documents in corpus: {len(corpus)}")
assert corpus, "Corpus is empty -- check df_notes_filtered has rows with TEXT"

### Graph embbedings the corpus using t-SNE and Spacy

In [ ]:
model1 = Word2Vec(corpus, min_count=1)
vocabs = model1.wv.key_to_index.keys()
new_v = np.array(list(vocabs))
tsne_plot(model1,new_v)

### Extract Entities with SciSpacy

In [ ]:
nlp_scispacy = spacy.load("en_core_sci_sm")

corpus_sci = []
for row in range(0, len(df_notes_filtered)):
    ents = nlp_scispacy(df_notes_filtered.iloc[row]['TEXT']).ents
    str_tokens = [t.text for t in ents]
    if str_tokens:
        corpus_sci.append(str_tokens)

print(f"Documents in corpus_sci: {len(corpus_sci)}")
assert corpus_sci, "corpus_sci is empty"

### Graph embbedings the corpus using t-SNE and SciSpacy

In [ ]:
model1 = Word2Vec(corpus_sci, min_count=1)
vocabs = model1.wv.key_to_index.keys()
new_v = np.array(list(vocabs))
tsne_plot(model1,new_v)

### Extract Entities using MedSciSpacy

In [ ]:
# medspaCy adds a PyRuSH sentencizer that clashes with the scispacy parser
# (both want to set token.sent_start, producing "ValueError [E043]").
# Since we only need entities here, disable the medspaCy sentencizer and let
# the parser do sentence splitting.
nlp_medspacy = medspacy.load("en_core_sci_sm", disable=["medspacy_pyrush"])
print("medspaCy pipeline:", nlp_medspacy.pipe_names)

corpus_med = []
for row in range(0, len(df_notes_filtered)):
    ents = nlp_medspacy(df_notes_filtered.iloc[row]['TEXT']).ents
    str_tokens = [t.text for t in ents]
    if str_tokens:
        corpus_med.append(str_tokens)

print(f"Documents in corpus_med: {len(corpus_med)}")
print("Sample entities from first note:", corpus_med[0][:20] if corpus_med else "empty")
assert corpus_med, "corpus_med is empty"